# Inference-Time Comparison: Qwen3-4B base vs. our improved QA model

Benchmarks **one model per run**. Workflow:
1. Run every cell with `MODEL_ID = BASE_MODEL_ID` (the default below). Note the printed
   summary at the end.
2. Edit the config cell so `MODEL_ID = IMPROVED_MODEL_ID` instead, then run every cell
   again. Note that summary too.
3. Compare the two printed summaries — that's the result.

Each run reports two things: **tokenization efficiency** (how many tokens this model's own
tokenizer needs to encode the exact same query + context) and **answer-generation time**
(measured `model.generate()` wall-clock, warm-up excluded).

Every prompt string, system prompt, and generation setting is copied verbatim from
`qwen3-4b-instruct-2507-test-split-inference.ipynb` (base model) and
`qwen3-4b-sinhala-qa-on-cpt.ipynb` (improved model) — each model automatically gets its own
real settings below, picked from `MODEL_ID`.

In [ ]:
# Installs the packages needed to load and run the model.
%uv pip install -q "transformers>=4.51,<5" accelerate safetensors huggingface_hub hf_transfer

In [ ]:
# Configuration. SET MODEL_ID HERE — swap which line is commented out, then re-run the whole
# notebook to benchmark the other model. Everything else below adapts automatically.
import os
import statistics
import time

import torch

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

BASE_MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
IMPROVED_MODEL_ID = "isji/qwen3-4b-sinhala-qa-cpt-v2-merged"

MODEL_ID = BASE_MODEL_ID
# MODEL_ID = IMPROVED_MODEL_ID

NO_ANSWER = "මෙම ප්‍රශ්නයට පිළිතුරු දීමට ප්‍රමාණවත් තොරතුරු නොමැත."

# ---- The query and retrieved context to benchmark on (same text every run) ----
QUERY = "ශ්‍රී ලංකාවේ ප්‍රාග් ඓතිහාසික යුගයේ විසූ මානවයාගේ ජීවන රටාව පිළිබඳව තොරතුරු ලබා ගැනීමට ඉවහල් වන සාධක බොහොමයක් ලැබී ඇති ස්ථානයක් ලෙස සැලකෙනුයේ ?"
CHUNKS = [
    "- වසර 3 800 ඡායාරූපය අංක 2.1 කළුතර දිස්ත්‍රික්කයට අයත් බුලත්සිංහල පාහියන්ගල පිහිටි මෙම ගල් ගුහාව මීට අවුරුදු 38 000 ඉහත කාලයක දී ප්‍රාග් ඓතිහාසික මානවයන්ගේ වාසස්ථානයක් ව පැවතිණි. පාහියන්ගල ලෙන ශ්‍රී ලංකාවේ පහතරට තෙත් කලාපයේ ජීවත් වූ ප්‍රාග් ඓතිහාසික මිනිසුන්ගේ ජීවිතය පිළිබඳ වැදගත් තොරතුරු අනාවරණය කර තිබේ. ජනාවාසවල මූලික ලක්ෂණ ශ්‍රී ලංකාවේ ප්‍රාග් ඓතිහාසික මානවයා රටේ පවතින එකිනෙකට වෙනස් පරිසර තත්ත්වවලට අනුවර්තනය වෙමින් ව්‍යාප්ත විය. ආහාර සඳහා අවශ්‍ය කරන ස්වාභාවික සම්පත් බහුල ප්‍රදේශවල වාසය කිරීමට ඔවුහු වැඩි කැමැත්තක් දැක්වූහ. පහතරට වැසි වනාන්තර ප්‍රදේශ, වියළි කලාපීය වනාන්තර, මුහුදුබඩ කළපු සහ විල්ලු ආශ්‍රිත ප්‍රදේශ, කඳුකර තණබිම් ආදී තැන්වල ඔවුන් වාසය කර ඇති බව සොයා ගෙන තිබේ. ජලය පහසුවෙන් ලබා ගත හැකි තැන් සහ ගල් මෙවලම් තනා ගැනීමට අවශ්‍ය කරන පාෂාණ වර්.",
    "ප්‍රමාණයෙන් සහ ආයත චතුරස්‍රාකාර හැඩයෙන් යුක්ත වූව කි. එහි ඉදිරිපස සහ පසුපස කොටස් බිත්තියකින් වෙන් කර තිබිණි. නිවසේ බිත්ති වරිච්චි මැටියෙන් ඉදි කොට තිබු අතර වහල ඉලුක් හෝ එවැනි දෙයකින් සෙවිලි කොට තිබෙන්නට ඇත. ජීවන රටාව ශ්‍රී ලංකාවේ පූර්ව ඓතිහාසික යුගයට අයත් ජීවන රටාව කෙබඳු ස්වරූපයකින් යුක්ත වූවක් දැ යි සම්පූර්ණ ව වටහා ගැනීමට ප්‍රමාණවත් සාක්ෂි තවමත් ලැබී නැත. දැනට වැඩි වශයෙන් කැනීම් කර තිබෙන්නේ සුසාන භූමි වන බැවින් ඒවායින් සොයා ගත හැකි ඒ කාලයේ සාමාන්‍ය ජන ජීවිතයට අයත් සාක්ෂි සීමිත ය. මේ පිළිබඳව යම් තරමකින් හෝ අදහසක් ලබා ගැනීමට ඉවහල් වන්නේ උඩරංචාමඩමේ නේවාසික ස්ථානයේ කළ කැණීමෙන් ලද තොරතුරු ය. උඩරංචාමඩමේ දී කණින ලද්දේ මීට අවුරුදු 3000 ට වඩා පැරණි කාලයේ ඉදි කරන ලදැයි විශ්වාස කරන නිවාසයකි. එහි ඇතුළත තිබී පැරණි වළං කටු රාශියක් සොයා ගැනිණි. ඒවා අතර පින්තාරු කළ වළඳකට අයත් කැබලි ගණනාවක් ද විය.",
    "තමන්ගේ එදිනෙදා අවශ්‍යතා සඳහා ඔවුන් තිරිවානා, කහඳ ආදි ගල්වලින් මෙවලම් තනා ගැනීමට පුරුදු වී සිටියහ. ශ්‍රී ලංකාවේ ප්‍රාග් ඉතිහාසය හැඳින්වීමට 'මධ්‍ය ශිලා යුගය' යන යෙදුම ද භාවිත වේ. 5. පසු කාලයක මෙරට විසූ ප්‍රාග් ඓතිහාසික මිනිසුන් ගොවිතැනට හුරු වූ බවට සාක්ෂි හමු වී තිබේ. දැනට ලැබී තිබෙන සාධක අනුව එම පරිවර්තනයන් ක්‍රිස්තු පූර්ව 2400 වන විට සම්පූර්ණත්වයකට පත් ව තිබිණි. 6. වළං සෑදීම, සුසාන භූමි භාවිතය සහ යකඩ ලෝහය භාවිත කිරීම පසු කාලයේ දී ආරම්භ විය. මෙම යුගය හැඳින්වෙන්නේ පූර්ව ඓතිහාසික යුගය යන නමිනි. 7. පූර්ව ඓතිහාසික යුගය වැදගත් කාලපරිච්ඡේදයක් වන්නේ ශ්‍රී ලංකාවේ ශිෂ්ටාචාරික වර්ධනයට අදාළ මුලික දැ එම යුගයෙන් ආරම්භ වීම නිසා ය. මෙරට මිනිසුන් ගම්මානවල ජීවත් වීම ආරම්භ කරන ලද්දේ මේ සමයේ දී ය.",
    "ක්‍රියාකාරකම 1 දැනට සොයාගෙන ඇති ශ්‍රී ලංකාවේ ප්‍රාග් ඓතිහාසික යුගයේ ජනාවාස ව්‍යාප්තිය සිතියමක ලකුණු කර නම් කරන්න. ඓතිහාසික යකඩ යුගය යනුවෙන් ද මුල් යකඩ යුගය යනුවෙන් ද හැඳින්වේ. එලෙස ම ස්ථීර ජනාවාස ඉදි කිරීම, කෘෂි කර්මාන්තය ඇරඹීම මෙකල දක්නට ලැබෙන තවත් පරිවර්තනයන් ය. පූර්ව ඓතිහාසික යුගය බිහිවීම ශ්‍රී ලංකාවේ පූර්ව ඓතිහාසික යුගයේ ආරම්භයත් ප්‍රාග් ඓතිහාසික යුගයේ අවසානයත් අතර සංක්‍රාන්තික සමය ගැන දැනට අපට තිබෙන දැනුම සීමිත ය. ප්‍රාග් ඓතිහාසික යුගයට අයත් ගල් මෙවලම් තාක්ෂණය සහ දඩයම ඇතුලුව ආහාර එකතු කිරීමේ යැපීම් ක්‍රමය යටපත් වෙමින් ඒ වෙනුවට ශාක ආහාර මත වැඩි වශයෙන් පදනම් වීම සහ ලෝහ භාවිතයට නැඹුරු වීම යන දෑ හිටිවන ම සිදු විය හැකි පරිවර්තනයක් නොවේ. අනෙක් අතට ඡායාරූපය අංක",
    "ප්‍රාග් ඓතිහාසික යුගයේ ජනාවාස සාහිත්‍ය මූලාශ්‍රය මගින් විස්තර කෙරෙන අතීත කාලයේ ආරම්භයට පෙර තිබූ යුගය පොදුවේ හැඳින්වෙන්නේ ප්‍රාග් ඓතිහාසික යුගය නමිනි. ශ්‍රී ලංකාවේ ප්‍රාග් ඓතිහාසික යුගයට සංස්කෘතික අවධි දෙකක් අයත් ය. එයින් පළමුවැන්න දීර්ඝ කාලයක් තිස්සේ පැවති ගල් යුගය යි. දෙවන යුගයට අයත් වන්නේ ශාක ආහාර මත යැපීම වෙත වැඩි නැඹුරුවක් තිබූ සහ ලෝහ භාවිතය සහ ස්ථිරවාසි ජනාවාස ආරම්භ වන කාලපරිච්ඡේදය යි. මේ අතරින් ගල්යුගය හැඳින්වීමට ප්‍රාග් ඓතිහාසික යුගය යන ව්‍යවහාරයත් දෙවන අවධිය හැඳින්වීමට පූර්ව ඵෙතිහාසික යුගය යන යෙදුමත් භාවිත කෙරේ. ජනාවාස ව්‍යාප්තිය ශ්‍රී ලංකාව ජනාවාස කරන ලද්දේ ආදි කාලීන හෝමෝසාපියන් මානවයා විසිනි. එකී මානවයා මෙරට විවිධ දේශගුණික කලාපවලට අනුවර්තනය වෙමින් පුළුල් භූගෝලීය ප්‍රදේශයක ව්‍යාප්ත විය. දඩයමින් සහ තැන තැන ඇවිද යමින් ආහාර එකතු කිරීම ඔවුන්ගේ ප්‍රධාන යැපීම් ක්‍රමය විය.",
]
CONTEXT_STR = "\n".join(CHUNKS)

REPETITION_PENALTY = 1.05
BASE_MAX_NEW_TOKENS = 320       # from qwen3-4b-instruct-2507-test-split-inference.ipynb
IMPROVED_MAX_NEW_TOKENS = 64    # from qwen3-4b-sinhala-qa-on-cpt.ipynb
IMPROVED_NUM_BEAMS = 6          # improved model's production decoding (beam search)
IMPROVED_LENGTH_PENALTY = 1.0

WARMUP_RUNS = 1     # discarded; absorbs one-time CUDA/cuDNN warm-up cost
TIMED_RUNS = 5       # repetitions actually timed, for a mean/median/stdev

print("Benchmarking MODEL_ID =", MODEL_ID)
print("Query  :", QUERY)
print("Chunks :", len(CHUNKS))

In [ ]:
# Logs in to Hugging Face — needed for the improved model, which is a private repo. Reads
# the token from the environment; never paste a token directly into a cell.
from huggingface_hub import login

_token = os.environ.get("HF_TOKEN")
if _token:
    login(token=_token, add_to_git_credential=False)
    print("Logged in to Hugging Face.")
else:
    print("No HF_TOKEN set — loading the private improved-model repo will fail.")

In [ ]:
# Picks the system prompt, output-length budget, and decoding strategy that match whichever
# MODEL_ID is set above — each model's own real settings, copied verbatim from its own
# notebook (see the intro). Nothing below this cell needs editing when you switch models.
BASE_SYSTEM_PROMPT = f"""You are a helpful Sinhala history question-answering assistant.

Your task is to answer the question using ONLY the information explicitly provided in the context.

Instructions:

- Read the entire context carefully before answering.
- Use only the information explicitly stated in the context.
- Do not use external knowledge, assumptions, or prior knowledge.
- Identify the exact information requested by the question.
- If the answer is found in multiple parts of the context, combine the relevant information into a single complete answer.
- Include only information that directly answers the question.
- Do not include additional facts, names, dates, or events unless they are required to answer the question.
- Match the person or entity named in the question exactly.
- Use evidence that contains both the requested entity and the requested attribute.
- Do not take a date or fact from a neighboring sentence about a different entity or event.
- Do not infer or guess information that is not explicitly stated, except for simple arithmetic explicitly requested by the question when all required values are stated in the context.
- For a duration question with explicit starting and ending years, subtract the starting year from the ending year and return the duration.
- If the answer cannot be found in the context, respond exactly with:
  "{NO_ANSWER}"
- Return only the final answer in natural Sinhala.
- Do not explain your reasoning.
- Do not mention passage numbers, page numbers, chapter names, grades, or any other source references.
- Answer in a single line. Do not add a preamble, a label, or quotation marks."""

IMPROVED_SYSTEM_PROMPT = f"""You are a helpful Sinhala history question-answering assistant.

Your task is to answer the question using ONLY the information explicitly provided in the context.

Instructions:

- Read the entire context carefully before answering.
- Use only the information explicitly stated in the context.
- Do not use external knowledge, assumptions, or prior knowledge.
- Identify the exact information requested by the question.
- If the answer is found in multiple parts of the context, combine the relevant information into a single complete answer.
- Include only information that directly answers the question.
- Do not include additional facts, names, dates, or events unless they are required to answer the question.
- Match the person or entity named in the question exactly.
- Use evidence that contains both the requested entity and the requested attribute.
- Do not take a date or fact from a neighboring sentence about a different entity or event.
- Do not infer or guess information that is not explicitly stated, except for simple arithmetic explicitly requested by the question when all required values are stated in the context.
- For a duration question with explicit starting and ending years, subtract the starting year from the ending year and return the duration.
- If the answer cannot be found in the context, respond exactly with:
  "{NO_ANSWER}"
- Return only the final answer in natural Sinhala.
- Do not explain your reasoning.
- Do not mention passage numbers, page numbers, chapter names, grades, or any other source references.
- Answer in a single line, then stop. Do not continue with any further text."""

if MODEL_ID == BASE_MODEL_ID:
    SYSTEM_PROMPT = BASE_SYSTEM_PROMPT
    MAX_NEW_TOKENS = BASE_MAX_NEW_TOKENS
    GENERATE_KWARGS = {"do_sample": False, "repetition_penalty": REPETITION_PENALTY}
elif MODEL_ID == IMPROVED_MODEL_ID:
    SYSTEM_PROMPT = IMPROVED_SYSTEM_PROMPT
    MAX_NEW_TOKENS = IMPROVED_MAX_NEW_TOKENS
    GENERATE_KWARGS = {
        "do_sample": False,
        "repetition_penalty": REPETITION_PENALTY,
        "num_beams": IMPROVED_NUM_BEAMS,
        "length_penalty": IMPROVED_LENGTH_PENALTY,
        "early_stopping": True,
    }
else:
    raise ValueError(f"Unrecognized MODEL_ID: {MODEL_ID!r}. Set it to BASE_MODEL_ID or IMPROVED_MODEL_ID in the config cell above.")

print("Settings picked for this run:")
print(f"  MAX_NEW_TOKENS : {MAX_NEW_TOKENS}")
print(f"  decoding       : {GENERATE_KWARGS}")

In [ ]:
# Loads the model and tokenizer set by MODEL_ID above.
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=model_dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()
model.config.pad_token_id = tokenizer.pad_token_id

# Qwen3's chat template supports a thinking toggle; turn it off when present so generation
# goes straight to the answer (matches both source notebooks).
supports_thinking = "enable_thinking" in (tokenizer.chat_template or "")
chat_kwargs = {"enable_thinking": False} if supports_thinking else {}

print(f"Loaded {MODEL_ID}")
print(f"  vocab size : {len(tokenizer):,}")
print(f"  dtype      : {model_dtype}  | device: {model.device}")

In [ ]:
# TOKENIZATION EFFICIENCY — renders the prompt through this model's own tokenizer and
# reports how many tokens it took. No generation yet, so this number is a pure, immediate
# measure of tokenizer efficiency for this exact query + context.
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"Context:\n\n{CONTEXT_STR}\n\nQuestion:\n\n{QUERY}\n\nAnswer:"},
]
prompt_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    **chat_kwargs,
).to(model.device)
prompt_token_count = prompt_ids["input_ids"].shape[-1]

print("=== Tokenization efficiency ===")
print(f"Model         : {MODEL_ID}")
print(f"Vocab size    : {len(tokenizer):,}")
print(f"Prompt tokens : {prompt_token_count}   (same query+context text every run — compare this number across models)")

In [ ]:
# ANSWER GENERATION — times model.generate() over WARMUP_RUNS (discarded) + TIMED_RUNS
# (measured) calls, then prints a full summary. torch.cuda.synchronize() brackets each timed
# call so the clock reflects actual GPU work, not just when the async call returns control.
# Copy the "Full summary" block below to compare against the other model's run.
def run_once():
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(
            **prompt_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            eos_token_id=model.generation_config.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
            **GENERATE_KWARGS,
        )
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    generated_ids = output_ids[0, prompt_token_count:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return elapsed, len(generated_ids), text


for _ in range(WARMUP_RUNS):
    run_once()

timings, output_lengths, last_text = [], [], ""
for _ in range(TIMED_RUNS):
    elapsed, n_tokens, last_text = run_once()
    timings.append(elapsed)
    output_lengths.append(n_tokens)

mean_time = statistics.mean(timings)
mean_tokens = statistics.mean(output_lengths)

print("=== Full summary (copy this to compare against the other model's run) ===")
print(f"Model          : {MODEL_ID}")
print(f"Vocab size     : {len(tokenizer):,}")
print(f"Decoding       : {GENERATE_KWARGS}")
print(f"Prompt tokens  : {prompt_token_count}")
print(f"Output tokens  : {mean_tokens:.1f}")
print(f"Mean time      : {mean_time:.3f}s  (median {statistics.median(timings):.3f}s, "
      f"stdev {statistics.stdev(timings):.3f}s, over {TIMED_RUNS} runs)")
print(f"Tokens/second  : {mean_tokens / mean_time:.1f}")
print(f"Sample answer  : {last_text.splitlines()[0][:150] if last_text else '(empty)'}")